In [37]:
import polars as pl

edges = pl.read_csv("datasets/edges.csv")
edges

osm_id,from_id,to_id,distance_m,fclass,oneway,maxspeed
i64,i64,i64,f64,str,i64,i64
23262895,0,1,841.23,"""residential""",0,0
23262948,2,3,611.05,"""residential""",0,0
23262982,4,5,489.21,"""residential""",0,0
23262989,6,7,713.1,"""residential""",0,20
23263232,8,9,254.74,"""tertiary""",0,0
…,…,…,…,…,…,…
1516535910,197487,519627,120.21,"""tertiary""",0,60
1516539003,749533,197487,12.78,"""tertiary""",0,0
1516540794,886788,886789,25.71,"""tertiary_link""",0,0


In [38]:
print(edges.filter(pl.col("distance_m") <= 0))


shape: (0, 7)
┌────────┬─────────┬───────┬────────────┬────────┬────────┬──────────┐
│ osm_id ┆ from_id ┆ to_id ┆ distance_m ┆ fclass ┆ oneway ┆ maxspeed │
│ ---    ┆ ---     ┆ ---   ┆ ---        ┆ ---    ┆ ---    ┆ ---      │
│ i64    ┆ i64     ┆ i64   ┆ f64        ┆ str    ┆ i64    ┆ i64      │
╞════════╪═════════╪═══════╪════════════╪════════╪════════╪══════════╡
└────────┴─────────┴───────┴────────────┴────────┴────────┴──────────┘


In [39]:
print(edges.filter(pl.col("maxspeed") < 0))

shape: (0, 7)
┌────────┬─────────┬───────┬────────────┬────────┬────────┬──────────┐
│ osm_id ┆ from_id ┆ to_id ┆ distance_m ┆ fclass ┆ oneway ┆ maxspeed │
│ ---    ┆ ---     ┆ ---   ┆ ---        ┆ ---    ┆ ---    ┆ ---      │
│ i64    ┆ i64     ┆ i64   ┆ f64        ┆ str    ┆ i64    ┆ i64      │
╞════════╪═════════╪═══════╪════════════╪════════╪════════╪══════════╡
└────────┴─────────┴───────┴────────────┴────────┴────────┴──────────┘


In [40]:
dup_edges = edges.group_by("osm_id").agg(pl.len().alias("cantidad")).filter(pl.col("cantidad") > 1).get_column("osm_id")
dup_edges

osm_id
i64


In [41]:
bucles = edges.filter(pl.col("from_id") == pl.col("to_id"))
bucles

osm_id,from_id,to_id,distance_m,fclass,oneway,maxspeed
i64,i64,i64,f64,str,i64,i64
29323193,662,662,98.31,"""secondary""",0,0
29323236,684,684,174.84,"""footway""",0,0
30425860,776,776,205.19,"""primary""",0,40
35279776,1333,1333,60.08,"""pedestrian""",0,0
35344976,1430,1430,130.12,"""residential""",0,0
…,…,…,…,…,…,…
1500850746,884966,884966,21.42,"""footway""",0,0
1500850747,884967,884967,26.06,"""footway""",0,0
1500852043,884976,884976,155.49,"""pedestrian""",0,0


## No se encontro.
* Distancias negativas o iguales a cero.
* Velocidades maximas negativas.
* Nodos repetidos.

## Si se encontro.
* Bucles o nodos con camino a si mismos.
* Velocidadaes maximas iguales a cero.

## Limpieza consistio.
* Eliminar los bucles del dataset.
* Reemplazar valores iguales a cero de velocidades maximas.

#### Limpieza

#### Eliminar bucles

In [42]:
df = edges.join(bucles, on="osm_id", how="anti")
df

osm_id,from_id,to_id,distance_m,fclass,oneway,maxspeed
i64,i64,i64,f64,str,i64,i64
23262895,0,1,841.23,"""residential""",0,0
23262948,2,3,611.05,"""residential""",0,0
23262982,4,5,489.21,"""residential""",0,0
23262989,6,7,713.1,"""residential""",0,20
23263232,8,9,254.74,"""tertiary""",0,0
…,…,…,…,…,…,…
1516535910,197487,519627,120.21,"""tertiary""",0,60
1516539003,749533,197487,12.78,"""tertiary""",0,0
1516540794,886788,886789,25.71,"""tertiary_link""",0,0


In [43]:
total = 588_485 - 584_512
total

3973

Podemos ver que se eliminaron los 3973 registros que eran nodos con bucles a si mismos.

#### Reemplazar velocidades maximas iguales a cero

In [44]:
fclass = (edges.select("fclass").unique().to_series().to_list())
print(fclass)

['tertiary', 'track_grade5', 'secondary', 'primary_link', 'service', 'path', 'living_street', 'track_grade1', 'unclassified', 'track_grade2', 'footway', 'bridleway', 'track_grade4', 'track_grade3', 'residential', 'tertiary_link', 'secondary_link', 'primary', 'busway', 'trunk', 'steps', 'pedestrian', 'track', 'motorway_link', 'motorway', 'unknown', 'cycleway', 'trunk_link']


Se creo un diccionario en base a los `fclass` existentes y se le asigno una velocidad maxima.
>VALOR de Velocidad Maxima: En base a la _"Ley 3988 Codigo de Transito y Reglamento"_ se hizo la inferencia segun los fclass y los tipos de carreteras mencionados en los articulos 113 y 114.

In [45]:
speed_map = {
    "motorway": 80,
    "motorway_link": 80,
    "trunk": 80,
    "trunk_link": 80,
    "primary": 80,
    "primary_link": 80,
    "secondary": 70,
    "secondary_link": 70,
    "tertiary": 70,
    "tertiary_link": 70,
    "residential": 40,
    "living_street": 20,
    "service": 20,
    "pedestrian": 10,
    "busway": 40,
    "footway": 10,
    "cycleway": 10,
    "path": 10,
    "steps": 5,
    "bridleway": 10,
    "track": 70,
    "track_grade1": 70,
    "track_grade2": 60,
    "track_grade3": 50,
    "track_grade4": 40,
    "track_grade5": 30,
    "unclassified": 40,
    "unknown": 40,
}

In [46]:
df = df.with_columns(
    pl.when(pl.col("maxspeed") == 0)
    .then(
        pl.col("fclass").replace(speed_map).cast(pl.Int64)
    )
    .otherwise(pl.col("maxspeed"))
    .alias("maxspeed")
)
df

osm_id,from_id,to_id,distance_m,fclass,oneway,maxspeed
i64,i64,i64,f64,str,i64,i64
23262895,0,1,841.23,"""residential""",0,40
23262948,2,3,611.05,"""residential""",0,40
23262982,4,5,489.21,"""residential""",0,40
23262989,6,7,713.1,"""residential""",0,20
23263232,8,9,254.74,"""tertiary""",0,70
…,…,…,…,…,…,…
1516535910,197487,519627,120.21,"""tertiary""",0,60
1516539003,749533,197487,12.78,"""tertiary""",0,70
1516540794,886788,886789,25.71,"""tertiary_link""",0,70


In [47]:
print(df.filter(pl.col("maxspeed") <= 0))

shape: (0, 7)
┌────────┬─────────┬───────┬────────────┬────────┬────────┬──────────┐
│ osm_id ┆ from_id ┆ to_id ┆ distance_m ┆ fclass ┆ oneway ┆ maxspeed │
│ ---    ┆ ---     ┆ ---   ┆ ---        ┆ ---    ┆ ---    ┆ ---      │
│ i64    ┆ i64     ┆ i64   ┆ f64        ┆ str    ┆ i64    ┆ i64      │
╞════════╪═════════╪═══════╪════════════╪════════╪════════╪══════════╡
└────────┴─────────┴───────┴────────────┴────────┴────────┴──────────┘


Como se puede ver, ahora no existen velocidades menores o iguales a cero

### Guardado del nuevo dataframe

#### Quitar columnas innecesarias
* Las columnas: `['osm_id','fclass']` ya no son necesarias para poder armar los grafos.
* Se agrego la columna `time` [s], que es el resultado de `distance_m / (maxspeed * 10 / 36)` [m/s].

In [48]:
df = df.drop(['osm_id','fclass'])
df

from_id,to_id,distance_m,oneway,maxspeed
i64,i64,f64,i64,i64
0,1,841.23,0,40
2,3,611.05,0,40
4,5,489.21,0,40
6,7,713.1,0,20
8,9,254.74,0,70
…,…,…,…,…
197487,519627,120.21,0,60
749533,197487,12.78,0,70
886788,886789,25.71,0,70


In [52]:
df = df.with_columns(
    (pl.col('distance_m') / (pl.col('maxspeed').replace(0, None) * 10 / 36))
    .round(2)
    .alias("time")
)
df

from_id,to_id,distance_m,oneway,maxspeed,time
i64,i64,f64,i64,i64,f64
0,1,841.23,0,40,75.71
2,3,611.05,0,40,54.99
4,5,489.21,0,40,44.03
6,7,713.1,0,20,128.36
8,9,254.74,0,70,13.1
…,…,…,…,…,…
197487,519627,120.21,0,60,7.21
749533,197487,12.78,0,70,0.66
886788,886789,25.71,0,70,1.32


#### Guardamos en un nuevo archivo CSV

In [53]:
df.write_csv("datasets/edges-clean.csv")

## Observaciones

En este caso estamos asumiendo 2 cosas:
1. La velocidad maxima de algunas carreteras.
2. Que los autos siempre van a la velocidad maxima de cada carretera.

> Estas suposiciones alteran completamente los resultados que se pueden obtener al hacer el analisis comparativo de distancia vs. tiempo en cuanto a pesos.